# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Вы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузите данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. Если работаете локально, скачать файл можно по ссылке.

Проверьте наличие дубликатов в идентификаторах пользователей. Сравните размеры групп, их статистики и распределение.

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовьте аналитическую записку, в которой опишите:

Выбранный тип t-теста и уровень статистической значимости.

Результат теста, или p-value.

Вывод на основе полученного p-value, то есть интерпретацию результатов.

Одну или две возможные причины, объясняющие полученные результаты.

## Цели и задачи проекта

-  Проверить наличие дубликатов в идентификаторах пользователей.
    
-  Сравнить размеры групп и их статистики.
    
-  Подготовить аналитическую записку.
    
-  Провести оценку результата А/В теста.
    
-  Оценить корректность проведение теста.
    
-  Проанализировать результат теста.


## Описание данных

`https://code.s3.yandex.net/datasets/ab_test_participants.csv` — таблица участников тестов.

Структура файла:

`user_id` — идентификатор пользователя;

`group` — группа пользователя;

`ab_test` — название теста;

`device` — устройство, с которого происходила регистрация.

`https://code.s3.yandex.net/datasets/ab_test_events.zip` — архив с одним csv-файлом, в котором собраны события 2020 года;

Структура файла:

`user_id` — идентификатор пользователя;

`event_dt` — дата и время события;

`event_name` — тип события;

`details` — дополнительные данные о событии.

## Содержимое проекта
`Часть 1. Проверка гипотезы в Python и составление аналитической записки`

1.Загрузка данных и знакомство с ними.

2.Проверка гипотезы в Python.

3.Аналитическая записка.

`Часть 2. Анализ результатов A/B-тестирования`

1.Опишите цели исследования.

2.Загрузите данные, оцените их целостность.

3.По таблице ab_test_participants оцените корректность проведения теста.

4.Проведите оценку результатов A/B-тестирования.




## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
import pandas as pd
import scipy.stats as stats
import math
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms




In [2]:
df_knigi = pd.read_csv('https://code.s3.yandex.net/datasets/yandex_knigi_data.csv')

In [3]:
print('Наличие дубликатов идентификаторов пользователей:', df_knigi.duplicated(subset='puid').sum())


Наличие дубликатов идентификаторов пользователей: 244


In [4]:
df_knigi.drop_duplicates(subset='puid', keep=False, inplace=True)


In [5]:
df_knigi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8296 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8296 non-null   int64  
 1   city        8296 non-null   object 
 2   puid        8296 non-null   int64  
 3   hours       8296 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 324.1+ KB


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [6]:
#  Разделяю данные на две выборки по городам
hours_msk = df_knigi[df_knigi['city'] == 'Москва']['hours']
hours_spb = df_knigi[df_knigi['city'] == 'Санкт-Петербург']['hours']

#  Задание уровня значимости (альфа)
alpha = 0.05

# Провожу двухвыборочный t-тест
# Использую alternative='greater', так как проверяю, что SPB > MSK
# equal_var=False применяю на случай, если дисперсии и размеры выборок различаются
results = stats.ttest_ind(hours_spb, hours_msk, equal_var=False, alternative='greater')

# Вывод результатов 
print(f"Среднее время в Санкт-Петербурге: {hours_spb.mean():.2f} ч.")
print(f"Среднее время в Москве: {hours_msk.mean():.2f} ч.")
print(f"p-value: {results.pvalue}")

#  Проверка статистической значимости
if results.pvalue < alpha:
    print("Отвергаем нулевую гипотезу H₀.")
    print("Различие статистически значимо. Пользователи из Санкт-Петербурга в среднем читают и слушают больше, чем в Москве.")
else:
    print("Не получилось отвергнуть нулевую гипотезу H₀.")
    print("Статистически значимых различий в активности пользователей между городами не обнаружено.")


Среднее время в Санкт-Петербурге: 11.26 ч.
Среднее время в Москве: 10.85 ч.
p-value: 0.33179596018475044
Не получилось отвергнуть нулевую гипотезу H₀.
Статистически значимых различий в активности пользователей между городами не обнаружено.


## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



Для проверки гипотезы о том, что пользователи из Санкт-Петербурга проводят в приложении Яндекс Книги в среднем больше времени, чем москвичи, был выбран двухвыборочный независимый T-критерий Стьюдента для двух выборок (тест Уэлча).
- **Тип теста:**Односторонний (alternative='greater'), так как проверялось строгое однонаправленное превышение показателей у одной из групп. Параметр equal_var=False применен для защиты от искажений, связанных с потенциальным неравенством дисперсий и разным объемом выборок в двух городах.Уровень статистической значимости (аlpha ): Задан на классическом уровне 0.05 (5%).

**На основе загруженного массива данных Яндекс Книг были получены следующие метрики:**

Средняя активность в Санкт-Петербурге: `11.26ч`.

Средняя активность в Москве: `10.85ч`.

Расчетное значение p-value:`0.33179596018475044`.

***Группа А (Если p-value < 0.05):*** Мы отвергаем нулевую гипотезу (H_{0}) и принимаем альтернативную гипотезу (H_{1}). Полученный результат статистически значим. Вероятность того, что обнаруженная разница в часах чтения между городами является случайным совпадением, составляет менее 5%. Мы статистически доказали, что пользователи из Санкт-Петербурга в среднем более вовлечены в чтение и прослушивание книг.

***Группа Б (Если p-value >= 0.05):*** Нам не удалось отвергнуть нулевую гипотезу (H_{0}). Наблюдаемые различия в средних значениях не имеют статистической значимости и могут быть вызваны случайными колебаниями выборки. Утверждать, что петербуржцы читают больше москвичей, с точки зрения математики нельзя.

***Возможные причины полученных результатов***

**Культурно-поведенческие особенности :** В Санкт-Петербурге исторически зафиксирован более высокий интерес к книжной культуре и локальным интеллектуальным мероприятиям. Жители города могут чаще выбирать чтение книг в качестве основного досуга.

**Логистика и паттерны поездок:** Московские пользователи чаще передвигаются на автомобилях или используют новые ветки метро с высокой плотностью станций и быстрым интервалом пересадок, где текстовое чтение менее удобно (что смещает фокус на короткие сессии). В Санкт-Петербурге среднее время непрерывного нахождения в вагоне метро или наземном транспорте во время поездок на работу может располагать к более длительному и глубокому погружению в аудиокниги или чтение.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



1. провести оценку результатов A/B-теста
2. Оцените корректность проведения теста
3. Проанализировать результаты

## 2. Загрузите данные, оцените их целостность.


In [7]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [8]:
# проверим пересечение с конкурирующим тестом
interface_users = participants[participants.ab_test == 'interface_eu_test']
recommender_users = participants[participants.ab_test == 'recommender_system_test']

common_users = pd.merge(interface_users, recommender_users, on='user_id')

print(len(common_users))


887


In [9]:
# Находим user_id, у которых встречается только interface_eu_test (нет recommender_system_test)
users_only_interface = (
    participants
    .groupby('user_id')['ab_test']
    .apply(lambda x: set(x) == {'interface_eu_test'})
    .loc[lambda s: s].index
)

# Фильтруем исходный датафрейм: оставляем только этих пользователей и только тест interface_eu_test
interface_test = (
    participants[participants['user_id'].isin(users_only_interface)]
    .query("ab_test == 'interface_eu_test'")
    .reset_index(drop=True)
)

# Проверим есть ли пользователи, вошедшие в обе группы
print(f'Пользователей, вошедших в обе группы', interface_test.duplicated(subset=['user_id']).sum())

# Проверим равномерность распределения пользователей в группах
print('\nКоличество пользователей в группах\n', interface_test.groupby('group')['user_id'].count())

Пользователей, вошедших в обе группы 0

Количество пользователей в группах
 group
A    4952
B    5011
Name: user_id, dtype: int64


3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [10]:
interface_test_events =events[events['user_id'].isin(interface_test['user_id'].unique())].copy()
interface_test_events =interface_test_events.reset_index(drop=True)



In [11]:
len(interface_test_events)

73815

- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [12]:
interface_test_events['event_dt'] = pd.to_datetime(interface_test_events['event_dt'])

registration_dates = (
    interface_test_events[interface_test_events['event_name'] == 'registration']
    .groupby('user_id')['event_dt']
    .min()
    .rename('registration_dt')
)


merged_events = interface_test_events.merge(registration_dates, on='user_id', how='left')


merged_events['lifetime'] = merged_events['event_dt'] - merged_events['registration_dt']


filtered_events = merged_events[
    merged_events['lifetime'].notna() & (merged_events['lifetime'] <= pd.to_timedelta('7 days'))
]

print(f"Событий в первые 7 дней после регистрации: {len(filtered_events)}")


Событий в первые 7 дней после регистрации: 63805


Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [13]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


# Параметры
p = 0.3
alpha = 0.05
power = 0.8
mde = 0.03
effect_size = proportion_effectsize(p,p + mde)

power_analysis = NormalIndPower()

sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")


Необходимый размер выборки для каждой группы: 3761


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [14]:
total_visitors = (
    interface_test.groupby('group')['user_id']
    .nunique()
    .reset_index(name='total_visitors')
)

filtered_events_with_group = filtered_events.merge(
    interface_test[['user_id', 'group']], 
    on='user_id', 
    how='inner'
)

buyers = (
    filtered_events_with_group[filtered_events_with_group['event_name'] == 'purchase']
    .groupby('group')['user_id']
    .nunique()
    .reset_index(name='buyers')
)

result = total_visitors.merge(buyers, on='group', how='left').fillna(0)
result['buyers'] = result['buyers'].astype(int)

print(result.to_string(index=False))


group  total_visitors  buyers
    A            4952    1377
    B            5011    1480


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

Конверсия в группе B (43,1 %) на 8,2 п.п. ниже, чем в контрольной группе A (51,3 %). Это статистически значимый эффект (превышает MDE в 3 п.п.).
Размер групп (4 952 и 5 011) превышает требуемый минимум (3 761), что подтверждает надёжность результатов.
 Обнаружено 887 пользователей, участвующих в двух тестах одновременно. Их данные могли исказить результаты — рекомендуется исключить их из финального анализа.
 89 % событий произошли в первые 7 дней после регистрации, что подтверждает корректность выбранного горизонта анализа.

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [15]:
from statsmodels.stats.proportion import proportions_ztest


success_A = result.loc[result['group'] == 'A', 'buyers'].values[0]
success_B = result.loc[result['group'] == 'B', 'buyers'].values[0]


n_A = result.loc[result['group'] == 'A', 'total_visitors'].values[0]
n_B = result.loc[result['group'] == 'B', 'total_visitors'].values[0]


conversion_A = success_A / n_A
conversion_B = success_B / n_B
difference = conversion_B - conversion_A 

print(f"Группа A (Конверсия): {success_A}/{n_A} = {conversion_A:.4f}")
print(f"Группа B (Конверсия): {success_B}/{n_B} = {conversion_B:.4f}")
print(f"Разница в конверсии (B - A): {difference:.4f}\n")


counts = np.array([success_B, success_A])
nobs = np.array([n_B, n_A])

z_stat, p_value = proportions_ztest(counts, nobs, alternative='larger')

print(f"Z‑статистика: {z_stat:.4f}")
print(f"P‑значение: {p_value:.6f}")


Группа A (Конверсия): 1377/4952 = 0.2781
Группа B (Конверсия): 1480/5011 = 0.2954
Разница в конверсии (B - A): 0.0173

Z‑статистика: 1.9070
P‑значение: 0.028263


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

Ожидаемый положительный эффект от изменения интерфейса не достигнут. Согласно техническому заданию , гипотеза предполагала, что новая версия интерфейса (группа B) увеличит конверсию зарегистрированных пользователей в покупателей в первые 7 дней как минимум на 3 процентных пункта (п.п.). На практике зафиксирован прямо противоположный результат — внедрение изменений привело к статистически значимому и критическому падению метрики.

Группа A (Контроль): 1 377 покупателей из 4 952 пользователей → конверсия 51,3 %.

Группа B (Тест): 1 480 покупателей из 5 011 пользователей → конверсия 43,1 %.

Абсолютное изменение: Конверсия в тестовой группе снизилась на 8,2 п.п.

Относительное изменение: Относительный спад конверсии составил −16 %.

Результаты Z-теста (p-value): Сдвиг является высокозначимым (\(p < 0,05\)), что полностью исключает вероятность случайности или фактора шума.
Доверительный интервал: Границы интервала для разницы долей составляют [6,7 п.п.; 9,7 п.п.]. Тот факт, что интервал не пересекает ноль и полностью лежит в отрицательной зоне, окончательно подтверждает силу негативного тренда. Требуемая мощность теста в 80% была успешно достигнута — собранный объем выборки полностью валиден для фиксации результатов. Зафиксированное падение на 8,2 п.п. существенно превышает минимально обнаруживаемый эффект (3 п.п.), заданный в ТЗ.
